In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.utils import to_categorical
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
df_dados = pd.read_csv('Clientes e Emprego2.csv')
df_dados.head()

,IDCREDITO,Duracao,HistoricoCredito,Proposito,Valor,Investimentos,Emprego,TempoParcelamento,EstadoCivil,FiadorTerceiros,...,Status,proposta,financiamentos,profissao,emprego,histcredito,fiador,residencia,estadocivil,investimentos
0,107,18,5,4,6458,2,2.0,2,4,3,...,0,veiculo novo,banco,altamente qualificado/empresario,>=7,todos pagos,nao,proria,Viuvo,<100\\n
1,108,12,2,4,6078,2,4.0,2,4,3,...,1,veiculo novo,nenhum,qualificado,4<=X<7,existentes/pagos,nao,proria,Viuvo,<100\\n
2,109,24,2,3,7721,1,1.0,1,1,3,...,1,moveis,nenhum,qualificado,<1,existentes/pagos,nao,proria,Solteiro,Desconhecido\\n
3,110,14,2,6,1410,3,2.0,1,3,3,...,1,abrir negocio,nenhum,qualificado,>=7,existentes/pagos,nao,proria,Divorciado,500<=X<1000
4,78,11,3,1,4771,2,NaN,2,4,3,...,1,reforma,nenhum,qualificado,NaN,Atrasos anteriores,nao,proria,Viuvo,<100\\n


In [3]:
# Excluindo colunas "desnecessárias ou substituídas.

df_dados.drop(columns = ['IDCREDITO', 'fiador', 'residencia', 'estadocivil', 'HistoricoCredito','Proposito', 'Investimentos', 'Emprego', 'EstadoCivil', 'FiadorTerceiros', 'ResidenciaDesde',
                      'OutrosFinanciamentos', 'Habitacao', 'EmprestimoExistente', 'Profissao', 'Dependentes', 'SocioEmpresa','TempoParcelamento'],
                        inplace = True)
df_dados.head()

,Duracao,Valor,Idade,Status,proposta,financiamentos,profissao,emprego,histcredito,investimentos
0,18,6458,39,0,veiculo novo,banco,altamente qualificado/empresario,>=7,todos pagos,<100\\n
1,12,6078,32,1,veiculo novo,nenhum,qualificado,4<=X<7,existentes/pagos,<100\\n
2,24,7721,30,1,moveis,nenhum,qualificado,<1,existentes/pagos,Desconhecido\\n
3,14,1410,35,1,abrir negocio,nenhum,qualificado,>=7,existentes/pagos,500<=X<1000
4,11,4771,51,1,reforma,nenhum,qualificado,NaN,Atrasos anteriores,<100\\n


In [4]:
# Colocando a coluna alvo no final.

df_dados['Status'] = df_dados.pop('Status')
df_dados.head()

,Duracao,Valor,Idade,proposta,financiamentos,profissao,emprego,histcredito,investimentos,Status
0,18,6458,39,veiculo novo,banco,altamente qualificado/empresario,>=7,todos pagos,<100\\n,0
1,12,6078,32,veiculo novo,nenhum,qualificado,4<=X<7,existentes/pagos,<100\\n,1
2,24,7721,30,moveis,nenhum,qualificado,<1,existentes/pagos,Desconhecido\\n,1
3,14,1410,35,abrir negocio,nenhum,qualificado,>=7,existentes/pagos,500<=X<1000,1
4,11,4771,51,reforma,nenhum,qualificado,NaN,Atrasos anteriores,<100\\n,1


In [5]:
df_dados.columns = ['Duracao', 'Valor', 'Idade', 'Proposta', 'Financiamentos', 'Profissao', 'Emprego', 'Historico_Credito', 'Investimentos','Status']
df_dados

,Duracao,Valor,Idade,Proposta,Financiamentos,Profissao,Emprego,Historico_Credito,Investimentos,Status
0,18,6458,39,veiculo novo,banco,altamente qualificado/empresario,>=7,todos pagos,<100\\n,0
1,12,6078,32,veiculo novo,nenhum,qualificado,4<=X<7,existentes/pagos,<100\\n,1
2,24,7721,30,moveis,nenhum,qualificado,<1,existentes/pagos,Desconhecido\\n,1
3,14,1410,35,abrir negocio,nenhum,qualificado,>=7,existentes/pagos,500<=X<1000,1
4,11,4771,51,reforma,nenhum,qualificado,NaN,Atrasos anteriores,<100\\n,1
...,...,...,...,...,...,...,...,...,...,...
995,15,1403,28,veiculo novo,nenhum,qualificado,1<=X<4,existentes/pagos,<100\\n,1
996,18,4297,40,moveis,nenhum,altamente qualificado/empresario,>=7,Atrasos anteriores,<100\\n,0
997,36,1977,40,educacao,nenhum,altamente qualificado/empresario,>=7,existentes/pagos,Desconhecido\\n,0
998,9,1136,32,educacao,nenhum,qualificado,>=7,Critico-outros creditos,>=1000,0


In [6]:
# Verificando valores nan.

df_dados.isnull().sum()
# Agrupando e substituindo.

emprego = df_dados.groupby('Emprego').size()
emprego

df_dados['Emprego'] = df_dados['Emprego'].fillna('1<=X<4 ')
df_dados['Emprego'].isnull().sum()

profissao = df_dados.groupby('Profissao').size()
profissao

df_dados['Profissao'] = df_dados['Profissao'].fillna('qualificado')
df_dados['Profissao'].isnull().sum()

histcred = df_dados.groupby('Historico_Credito').size()
histcred

df_dados['Historico_Credito'] = df_dados['Historico_Credito'].fillna('existentes/pagos')
df_dados['Historico_Credito'].isnull().sum()

np.int64(0)

In [7]:
previsores = df_dados.iloc[:, 0:9].values
classe = df_dados.iloc[:,9].values

label = LabelEncoder()

previsores[:, 3] = label.fit_transform(previsores[:, 3])
previsores[:, 4] = label.fit_transform(previsores[:, 4])
previsores[:, 5] = label.fit_transform(previsores[:, 5])
previsores[:, 6] = label.fit_transform(previsores[:, 6])
previsores[:, 7] = label.fit_transform(previsores[:, 7])
previsores[:, 8] = label.fit_transform(previsores[:, 8])


previsores = previsores.astype('float32') 

# Se a sua 'classe' também veio de texto ou é objeto:
classe = label.fit_transform(classe).astype('float32')

In [8]:
# Dividindo a base de dados em treino e teste.

X_treino, X_teste, Y_treino, Y_teste = train_test_split(previsores, classe, test_size= 0.3, random_state=0)

In [9]:
# Criação do modelo neural com o comando Sequential.
modelo = Sequential()

# A definição da entrada é feita fora das camadas.
modelo.add(Input(shape = (9,)))

# Primeira camada oculta com 5 neurônios.
modelo.add(Dense(units = 5, activation= 'relu'))

# Segunda camada oculta
modelo.add(Dense(units = 4, activation='relu'))

# Terceira camada oculta / Função softmax por que temos um problema de classificação com mais de duas classes.
modelo.add(Dense(units=1, activation= 'sigmoid'))

In [10]:
# Visualização da estrutura da rede neural.

modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 5)              │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79 (316.00 B)

 Trainable params: 79 (316.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Configuração dos parâmetros da rede neural (adam = atualiza os pesos / loss = calcula o erro). 

modelo.compile(optimizer='adam', loss = 'binary_crossentropy',
               metrics = ['accuracy'])

# Treinamento dividindo a base de treinamento em uma porção de validação.
modelo.fit(X_treino, Y_treino, epochs = 1000, validation_data = (X_teste, Y_teste))

Epoch 1/1000
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2957 - loss: 1002.5663 - val_accuracy: 0.3100 - val_loss: 898.3819
Epoch 2/1000
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2957 - loss: 699.3252 - val_accuracy: 0.3100 - val_loss: 584.1882
Epoch 3/1000
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2986 - loss: 414.0581 - val_accuracy: 0.3133 - val_loss: 285.2722
Epoch 4/1000
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3786 - loss: 131.7400 - val_accuracy: 0.6133 - val_loss: 12.8675
Epoch 5/1000
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7086 - loss: 18.8192 - val_accuracy: 0.6200 - val_loss: 10.0527
Epoch 6/1000
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6714 - loss: 9.7806 - val_accuracy: 0.6667 - val_loss: 9.1676
Epoch 7/1000
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6714 - loss: 8.9300 - val_accuracy: 0.6667 - val_loss: 8.9839
Epoch 8/1000
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6800 - loss: 8.2222 

In [12]:
previsoes = modelo.predict(X_teste)
previsoes = (previsoes > 0.5)
previsoes

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


array([[False],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [False],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [False],
       [False],
       [False],
       [ True],
       [

In [13]:
acerto = accuracy_score(Y_teste, previsoes)
acerto

0.7